In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_ViT
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from network import HierarchicalConvEmbedding, TransformerBlock, AddPositionEmbedding
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [4]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [5]:
model=load_ViT()

In [6]:
model.evaluate(x_test,y_test_onehot)

125/125 [==============================] - 5s 17ms/step - loss: 0.2587 - accuracy: 0.9237


[0.2586666941642761, 0.9237499833106995]

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("dense_16").output,
    outputs=model.output
)

In [7]:
l_label = [2,3,4,5,6,7,8,9,10,13]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_1/ViT-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_1/ViT-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_1/ViT-8/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data_1/ViT-8/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data_1/ViT-8/layer_cache/occ/"+str(i))

[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/base
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/gauss/0
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/salt/0
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/move/0
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/occ/0
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/gauss/1
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/salt/1
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/move/1
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/occ/1
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/gauss/2
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/salt/2
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/move/2
[SKIP] All layer files already exist in D:/Data_1/ViT-8/layer_cache/occ/2
[SKIP] All layer files alre

In [10]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "ViT_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data_1/ViT-8/layer_cache/attack/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_1/ViT-8/layer_cache/attack/0
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_1: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_2: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_3: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_4: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_5: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_6: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_7: outputs (20000, 8192), labels (20000, 1)
[Saved] dense_16: outputs (20000, 128), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_1/ViT-8/layer_cache/attack/1
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_bl

In [11]:
CACHE_DIR = "./ViT-8/w_and_b_cache"

In [12]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_1/ViT-8/layer_cache/base")

In [13]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_1/ViT-8/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_1/ViT-8/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.61001093 0.59601304 0.41180211 0.61306486]
Layer 1
accuracy: [0.61623938 0.59065285 0.46687709 0.60524947]
Layer 2
accuracy: [0.54549682 0.60001928 0.51409758 0.62676857]
Layer 3
accuracy: [0.5559351  0.60803175 0.60775498 0.68040682]
Layer 4
accuracy: [0.54809596 0.61062123 0.66242331 0.60969132]
Layer 5
accuracy: [0.61478999 0.66846707 0.70172882 0.66459774]
Layer 6
accuracy: [0.72641152 0.79887649 0.74319163 0.80821401]
Layer 7
accuracy: [0.9253565  0.93524935 0.94344342 0.94148671]
Layer 8
accuracy: [0.98626579 0.981256   0.98725797 0.96852523]
Layer 9
accuracy: [0.94609853 0.97223624 0.91770841 0.95318158]


In [16]:
base

array([[0.61001093, 0.59601304, 0.41180211, 0.61306486],
       [0.61623938, 0.59065285, 0.46687709, 0.60524947],
       [0.54549682, 0.60001928, 0.51409758, 0.62676857],
       [0.5559351 , 0.60803175, 0.60775498, 0.68040682],
       [0.54809596, 0.61062123, 0.66242331, 0.60969132],
       [0.61478999, 0.66846707, 0.70172882, 0.66459774],
       [0.72641152, 0.79887649, 0.74319163, 0.80821401],
       [0.9253565 , 0.93524935, 0.94344342, 0.94148671],
       [0.98626579, 0.981256  , 0.98725797, 0.96852523],
       [0.94609853, 0.97223624, 0.91770841, 0.95318158],
       [1.        , 1.        , 1.        , 1.        ]])

In [17]:
attack = np.zeros((len(layer_list),4))
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/ViT-8/layer_cache/attack/"+str(i))
attack = attack/5
attack_final = np.zeros(4)
for i in range(5):
    DIR = ATTACK_dir + str(i) + '/ViT_pgd.npy'
    x_noise = np.load(DIR)
    attack_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.61627616 0.56378037 0.41482757 0.59309821]
Layer 1
accuracy: [0.6400485  0.57233837 0.51719919 0.57186971]
Layer 2
accuracy: [0.64991388 0.55638162 0.64928439 0.60414151]
Layer 3
accuracy: [0.6000696  0.56152144 0.64878517 0.54449592]
Layer 4
accuracy: [0.6099963  0.56355664 0.5549408  0.52501445]
Layer 5
accuracy: [0.56760376 0.54529854 0.53189431 0.51709414]
Layer 6
accuracy: [0.49871985 0.40183113 0.52943272 0.23622369]
Layer 7
accuracy: [0.24823968 0.18560144 0.34883831 0.1059604 ]
Layer 8
accuracy: [0.23798453 0.15085054 0.33784409 0.09721255]
Layer 9
accuracy: [0.24048519 0.15109492 0.33808919 0.09669866]
Layer 0
accuracy: [0.61529834 0.56503562 0.41610073 0.5928403 ]
Layer 1
accuracy: [0.6399741  0.56895303 0.51773768 0.57465107]
Layer 2
accuracy: [0.65064312 0.55811994 0.64698686 0.60263125]
Layer 3
accuracy: [0.59448757 0.566831   0.64856036 0.54824652]
Layer 4
accuracy: [0.61346438 0.56330722 0.55439392 0.52763156]
Layer 5
accuracy: [0.57050973 0.54535615

In [18]:
attack

array([[0.61617839, 0.56412685, 0.41568266, 0.59253976],
       [0.64099542, 0.57005687, 0.51701426, 0.57307599],
       [0.64807646, 0.55719029, 0.64765405, 0.60450384],
       [0.5975513 , 0.56420907, 0.64960106, 0.54416873],
       [0.61253202, 0.56291113, 0.55726457, 0.52579893],
       [0.5696934 , 0.5438549 , 0.53366244, 0.5203083 ],
       [0.49802164, 0.39744881, 0.52816673, 0.23775327],
       [0.24817772, 0.18807087, 0.3502979 , 0.10312718],
       [0.23740375, 0.1525277 , 0.33772647, 0.09429615],
       [0.23986523, 0.15232557, 0.33787665, 0.09354862],
       [0.23845   , 0.15175   , 0.33790001, 0.09335   ]])

In [19]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/ViT-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.67327696 0.3481674  0.62004277 0.48233158]
Layer 1
accuracy: [0.74666226 0.51664997 0.74722952 0.57618576]
Layer 2
accuracy: [0.7501588  0.72782826 0.7499597  0.71034465]
Layer 3
accuracy: [0.7501588  0.74533996 0.7499597  0.73584014]
Layer 4
accuracy: [0.7501588  0.74812124 0.7499597  0.72512974]
Layer 5
accuracy: [0.7501588  0.74988927 0.7499597  0.72740167]
Layer 6
accuracy: [0.7501588  0.74987327 0.7499597  0.66858574]
Layer 7
accuracy: [0.73937963 0.74690661 0.7499597  0.67251269]
Layer 8
accuracy: [0.72888098 0.74407884 0.74996386 0.65239068]
Layer 9
accuracy: [0.73093942 0.72066924 0.74795684 0.51587444]
Layer 0
accuracy: [0.67400692 0.3520596  0.61396235 0.47540217]
Layer 1
accuracy: [0.7458401  0.51314286 0.74796482 0.57442677]
Layer 2
accuracy: [0.7501588  0.73077273 0.7499597  0.71235612]
Layer 3
accuracy: [0.7501588  0.74509789 0.7499597  0.73506403]
Layer 4
accuracy: [0.7501588  0.75038704 0.7499597  0.72384653]
Layer 5
accuracy: [0.7501588  0.75063678

In [20]:
gauss

array([[0.67661757, 0.3494139 , 0.61528346, 0.47852012],
       [0.74637962, 0.51619748, 0.74776908, 0.57570919],
       [0.7501588 , 0.72940161, 0.7499597 , 0.71072983],
       [0.7501588 , 0.74537839, 0.7499597 , 0.73364147],
       [0.7501588 , 0.7485271 , 0.7499597 , 0.7241586 ],
       [0.7501588 , 0.75021364, 0.7499597 , 0.72833864],
       [0.75013342, 0.74973879, 0.7499597 , 0.66914734],
       [0.73901914, 0.74976803, 0.74993519, 0.6779114 ],
       [0.73038395, 0.74515376, 0.74989028, 0.65809599],
       [0.73288322, 0.7224592 , 0.74881509, 0.51562307],
       [0.65064999, 0.72405   , 0.74395   , 0.480775  ]])

In [21]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/ViT-8/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.65345083 0.40888228 0.53444341 0.48508224]
Layer 1
accuracy: [0.72324077 0.46017464 0.72924866 0.49958207]
Layer 2
accuracy: [0.74989599 0.70840354 0.7499597  0.68995771]
Layer 3
accuracy: [0.74990422 0.73492368 0.7499597  0.72962757]
Layer 4
accuracy: [0.74940483 0.74211725 0.7499597  0.72284007]
Layer 5
accuracy: [0.74364281 0.75385397 0.7499597  0.73348467]
Layer 6
accuracy: [0.73866167 0.76539221 0.7499597  0.73799113]
Layer 7
accuracy: [0.5782589  0.78863835 0.75021377 0.75990887]
Layer 8
accuracy: [0.58670806 0.79391199 0.76068299 0.7560233 ]
Layer 9
accuracy: [0.52376822 0.76078781 0.75694717 0.70847783]
Layer 0
accuracy: [0.65099218 0.41330119 0.53519799 0.49077372]
Layer 1
accuracy: [0.71950254 0.46345706 0.73045716 0.50235955]
Layer 2
accuracy: [0.74966619 0.70678102 0.7499597  0.69058542]
Layer 3
accuracy: [0.7501588  0.73758875 0.7499597  0.73006298]
Layer 4
accuracy: [0.74990422 0.73935586 0.7499597  0.7212238 ]
Layer 5
accuracy: [0.7471497  0.75413432

In [22]:
salt

array([[0.65076673, 0.40846082, 0.53372245, 0.48720956],
       [0.72180176, 0.46341482, 0.72997327, 0.50142892],
       [0.74973382, 0.70804321, 0.74993453, 0.68915955],
       [0.75013244, 0.73511375, 0.7499597 , 0.7269987 ],
       [0.7499061 , 0.73968801, 0.7499597 , 0.71721245],
       [0.74665514, 0.75277906, 0.74998488, 0.73211909],
       [0.74024705, 0.76117812, 0.74998488, 0.73497274],
       [0.57836251, 0.78231493, 0.75096294, 0.75691993],
       [0.58517953, 0.78367672, 0.76104276, 0.750698  ],
       [0.51950824, 0.75578566, 0.75662116, 0.71803275],
       [0.498525  , 0.77747501, 0.7882    , 0.753675  ]])

In [23]:
move = np.zeros((len(layer_list),4))
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/ViT-8/layer_cache/move/"+str(i))
move = move/10
move_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/move.npy'
    x_noise = np.load(DIR)
    move_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.63553364 0.72463454 0.41978815 0.69959747]
Layer 1
accuracy: [0.62352071 0.74514212 0.41317603 0.71311021]
Layer 2
accuracy: [0.45030742 0.68801273 0.3544792  0.68298778]
Layer 3
accuracy: [0.39619449 0.65809212 0.40218102 0.71536355]
Layer 4
accuracy: [0.36918877 0.74586577 0.38649299 0.71411483]
Layer 5
accuracy: [0.44319254 0.74433338 0.42301122 0.72415754]
Layer 6
accuracy: [0.50317663 0.74935041 0.43397265 0.72062075]
Layer 7
accuracy: [0.77066369 0.7488279  0.55165471 0.72772427]
Layer 8
accuracy: [0.80736597 0.75137167 0.64902999 0.73703324]
Layer 9
accuracy: [0.79391465 0.75041353 0.6413359  0.74202253]
Layer 0
accuracy: [0.63357655 0.72936669 0.42131641 0.70135831]
Layer 1
accuracy: [0.62328781 0.74638712 0.40891611 0.71413545]
Layer 2
accuracy: [0.45675915 0.68409273 0.35647379 0.68033928]
Layer 3
accuracy: [0.39943082 0.66017683 0.40294798 0.71734712]
Layer 4
accuracy: [0.37737836 0.74159443 0.37970475 0.71647304]
Layer 5
accuracy: [0.44966182 0.74112266

In [24]:
move

array([[0.63456651, 0.7273563 , 0.41964496, 0.70020764],
       [0.62412486, 0.74628908, 0.41229025, 0.71376664],
       [0.45533699, 0.68521645, 0.35428247, 0.67883024],
       [0.3948725 , 0.65661779, 0.40422895, 0.7125097 ],
       [0.37272388, 0.74332568, 0.38636382, 0.71080974],
       [0.44458059, 0.74023579, 0.41316452, 0.71943692],
       [0.50164325, 0.74819598, 0.42627067, 0.71984905],
       [0.76801203, 0.74654243, 0.55067542, 0.72419976],
       [0.81155931, 0.75026884, 0.64638164, 0.73402196],
       [0.79548531, 0.74928511, 0.63838541, 0.74241797],
       [0.81685   , 0.74027499, 0.684675  , 0.71837501]])

In [25]:
occ = np.zeros((len(layer_list),4))
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/ViT-8/layer_cache/occ/"+str(i))
occ = occ/10
occ_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/occ.npy'
    x_noise = np.load(DIR)
    occ_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.57533993 0.54360671 0.38306019 0.56450517]
Layer 1
accuracy: [0.65428079 0.64097106 0.49417026 0.61904854]
Layer 2
accuracy: [0.64010106 0.69430513 0.55841405 0.66822393]
Layer 3
accuracy: [0.70797846 0.73578944 0.63800434 0.73341699]
Layer 4
accuracy: [0.70596131 0.74407447 0.64242659 0.73285937]
Layer 5
accuracy: [0.73124463 0.74739072 0.67138444 0.73798507]
Layer 6
accuracy: [0.75247207 0.7618888  0.69265992 0.73213929]
Layer 7
accuracy: [0.80356419 0.78017679 0.78037694 0.72952173]
Layer 8
accuracy: [0.83231907 0.8024401  0.80514028 0.7175698 ]
Layer 9
accuracy: [0.80176808 0.79315646 0.73736625 0.68449875]
Layer 0
accuracy: [0.58024688 0.5415831  0.382573   0.57148112]
Layer 1
accuracy: [0.6580671  0.63442614 0.49844774 0.62372113]
Layer 2
accuracy: [0.63503015 0.6996322  0.56651831 0.66497168]
Layer 3
accuracy: [0.71326127 0.74018715 0.64399242 0.73043023]
Layer 4
accuracy: [0.70665091 0.74257901 0.64224126 0.73055672]
Layer 5
accuracy: [0.73339096 0.75332313

In [26]:
occ

array([[0.57654315, 0.54452609, 0.3806467 , 0.5656379 ],
       [0.6587794 , 0.63849105, 0.49804444, 0.62155282],
       [0.64152976, 0.69714258, 0.56198644, 0.66633978],
       [0.71317518, 0.73933001, 0.6410174 , 0.72970506],
       [0.70981752, 0.74372712, 0.64505438, 0.72874014],
       [0.73168655, 0.75019992, 0.67519705, 0.73659421],
       [0.75282133, 0.76052298, 0.69613004, 0.72971174],
       [0.7999664 , 0.78071816, 0.78149969, 0.71586484],
       [0.8258027 , 0.79695772, 0.80573043, 0.70874782],
       [0.79323972, 0.79242596, 0.74358631, 0.67538197],
       [0.84045001, 0.80642499, 0.80475   , 0.67742501]])

In [27]:
SAVE_FILE='ViT-8.pkl'

In [28]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [29]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6):
    """
    Input:
        matrix_dict: {name: np.ndarray(m, n)}, 6 matrices total
    Output:
        stats: {
            name: {
                "mean": (m,),
                "min":  (m,),
                "max":  (m,)
            }, ...
        }
    For each component i, compute statistics along axis=1 (across n samples):
        mean[i] = mean(X[i, :])
        min[i]  = min(X[i, :])
        max[i]  = max(X[i, :])
    """
    if check_num is not None and len(matrix_dict) != check_num:
        raise ValueError(f"Expected {check_num} matrices, got {len(matrix_dict)}")

    # shape consistency
    shapes = [np.asarray(v).shape for v in matrix_dict.values()]
    if len(set(shapes)) != 1:
        raise ValueError(f"Matrix shapes inconsistent: {shapes}")

    stats = {}
    for name, X in matrix_dict.items():
        X = np.asarray(X)
        stats[name] = {
            "mean": X.mean(axis=1),
            "std":  X.std(axis=1),
            "min":  X.min(axis=1),
            "max":  X.max(axis=1),
        }
    return stats

In [31]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [32]:
mean_var

{'base': {'mean': array([0.55772274, 0.5697547 , 0.57159556, 0.61303216, 0.60770795,
         0.66239591, 0.76917341, 0.93638399, 0.98082625, 0.94730619,
         1.        ]),
  'std': array([0.08449228, 0.06008585, 0.04426943, 0.04430652, 0.04049624,
         0.03104476, 0.03503596, 0.00704922, 0.00745747, 0.01958006,
         0.        ]),
  'min': array([0.41180211, 0.46687709, 0.51409758, 0.5559351 , 0.54809596,
         0.61478999, 0.72641152, 0.9253565 , 0.96852523, 0.91770841,
         1.        ]),
  'max': array([0.61306486, 0.61623938, 0.62676857, 0.68040682, 0.66242331,
         0.70172882, 0.80821401, 0.94344342, 0.98725797, 0.97223624,
         1.        ])},
 'attack': {'mean': array([0.54713191, 0.57528564, 0.61435616, 0.58888254, 0.56462666,
         0.54187976, 0.41534761, 0.22241842, 0.20548852, 0.20590402,
         0.2053625 ]),
  'std': array([0.07809773, 0.04400441, 0.03745268, 0.03990611, 0.03106308,
         0.01809939, 0.11338349, 0.09003729, 0.09175245, 0.0922